[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobMulla/torchtpu-inference-workshop/blob/main/notebooks/00_all_three_demos.ipynb)

# Inference with TorchTPU: the three demos in one notebook

All three demos of the workshop on one TPU runtime, top to bottom (*Runtime → Run all* works):

1. **TorchTPU on one chip:** the device line, eager, DeferAndFuse, `torch.compile`, and one step of each under the profiler.
2. **SGLang on TorchTPU:** a server on one chip, one request, a burst of sixteen, the prefix cache.
3. **vLLM on TorchTPU:** the same with vLLM.

The live demos ran Qwen3-32B over four chips; here it is `Qwen/Qwen3-0.6B` on one. Part 1 runs as a small script in a subprocess and each engine lives in a Python 3.12 environment of its own and is stopped before the next starts, so the notebook's kernel never holds the chip. About twenty minutes the first time (two installs, one model download).

**Runtime:** in Colab, *Runtime → Change runtime type → TPU*.

## Part 1 · TorchTPU: the TPU is a PyTorch device

![TorchTPU makes the TPU a PyTorch device: eager, DeferAndFuse, torch.compile](https://raw.githubusercontent.com/RobMulla/torchtpu-inference-workshop/main/notebooks/img/slide_03.jpg)

### 1.1 Install

`torch_tpu` from PyPI. One package; nothing else to set up.

In [ ]:
import importlib.util, os

if importlib.util.find_spec("google.colab"):  # Colab's TPU runtime: one chip, no metadata server
    for k, v in {"TPU_SKIP_MDS_QUERY": "1", "TPU_CHIPS_PER_HOST_BOUNDS": "1,1,1",
                 "TPU_HOST_BOUNDS": "1,1,1", "TPU_WORKER_ID": "0", "TPU_WORKER_HOSTNAMES": "localhost"}.items():
        os.environ.setdefault(k, v)  # inherited by the scripts and servers started below

import importlib.util, importlib.metadata as md, sys

if importlib.util.find_spec("torch_tpu") is None:
    %pip install -q torch_tpu==0.2.0 --extra-index-url https://download.pytorch.org/whl/cpu

for p in ["torch", "torch_tpu", "libtpu"]:
    print(f"{p:10s} {md.version(p)}")

### 1.2 The script

The live demo's model: a 12-block MLP and a forward pass in a loop, which is what an engine runs thousands of times. `device = "tpu"` is the whole change from a CPU script; `import torch` registers the device. The mode is the argument: eager runs one op at a time; `defer` sets `TORCH_TPU_DEFER_AND_FUSE=1`, TorchTPU's optimized eager mode (ops held back and run in fused blocks, still eager, marked experimental); `compiled` adds the one `torch.compile` line. Each run times 300 steps, then records five under the standard `torch.profiler` with the TPU activity on. On a TPU VM `import torch` alone registers the `tpu` device; Colab's sandbox hides the PCI bus TorchTPU scans for the chip, so the script registers the device explicitly (two lines that do nothing on a VM).


In [ ]:
%%writefile one_chip.py
# The live demo's model, one way per call:
#   python one_chip.py eager | defer | compiled
# 300 timed steps, then five steps under the profiler to traces/<mode>.
import os, sys, time, warnings

# Only the step times and the trace path: libtpu and the profiler log a lot of INFO.
warnings.filterwarnings("ignore")
os.environ.setdefault("TPU_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TPU_STDERR_LOG_LEVEL", "2")
os.environ.setdefault("TORCH_CPP_LOG_LEVEL", "ERROR")

mode = sys.argv[1]
if mode == "defer":  # the optimized eager mode, as in the live demo
    os.environ["TORCH_TPU_DEFER_AND_FUSE"] = "1"

import torch  # TorchTPU loads with torch; on a TPU VM that alone registers the "tpu" device
from torch_tpu import _loader
if not _loader._LOADED:  # a sandbox that hides /sys/bus/pci (Colab's): the scan found no TPU, register it anyway
    _loader._init_device("tpu")
from torch import nn
from torch.profiler import ProfilerActivity, profile, tensorboard_trace_handler
from torch.tpu.profiler import TpuProfilerConfig

device = "tpu"  # the whole change from a CPU script
model = nn.Sequential(*[
    nn.Sequential(nn.LayerNorm(1024),
                  nn.Linear(1024, 4096), nn.GELU(),
                  nn.Linear(4096, 1024))
    for _ in range(12)
]).to(device)
x = torch.randn(16, 512, 1024, device=device)
if mode == "compiled":  # one more line: the whole step as one program
    model = torch.compile(model, backend="tpu")

with torch.no_grad():
    for i in range(1, 301):
        t0 = time.perf_counter()
        y = model(x)
        torch.accelerator.synchronize()  # wait for the chip
        ms = (time.perf_counter() - t0) * 1000
        if i == 1 or i % 100 == 0:
            print(f"{mode} step {i}: {ms:.1f} ms", flush=True)

    trace_dir = os.path.abspath(f"traces/{mode}")
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.TPU],
        experimental_config=TpuProfilerConfig(run_dir=trace_dir),
        schedule=torch.profiler.schedule(wait=0, warmup=0, active=5, repeat=1),
        on_trace_ready=tensorboard_trace_handler(trace_dir),
    ) as prof:
        for i in range(1, 6):
            y = model(x)
            torch.accelerator.synchronize()
            prof.step()
print("trace:", trace_dir, flush=True)

### 1.3 Eager

Read the step time off the output.

In [ ]:
!{sys.executable} one_chip.py eager 2>&1 | grep -vE "^(I[0-9]{4}|W[0-9]{4}|E[0-9]{4}|USDT:|WARNING: All log|\[W[0-9]{4}|Successfully renamed|Registered Python module|Device type: tpu|Initializing TPU distributed|num_cache|num_compilations|peak_compilation|=== Source|learning/)|UserWarning|_tcmalloc|experimental_config=|schedule=torch"


### 1.4 DeferAndFuse

Same script, one setting. It says up front that the flag is experimental, then the same loop runs.

In [ ]:
!{sys.executable} one_chip.py defer 2>&1 | grep -vE "^(I[0-9]{4}|W[0-9]{4}|E[0-9]{4}|USDT:|WARNING: All log|\[W[0-9]{4}|Successfully renamed|Registered Python module|Device type: tpu|Initializing TPU distributed|num_cache|num_compilations|peak_compilation|=== Source|learning/)|UserWarning|_tcmalloc|experimental_config=|schedule=torch"


### 1.5 Compiled

The first step pays the compile; then compare the step time with the two above.

In [ ]:
!{sys.executable} one_chip.py compiled 2>&1 | grep -vE "^(I[0-9]{4}|W[0-9]{4}|E[0-9]{4}|USDT:|WARNING: All log|\[W[0-9]{4}|Successfully renamed|Registered Python module|Device type: tpu|Initializing TPU distributed|num_cache|num_compilations|peak_compilation|=== Source|learning/)|UserWarning|_tcmalloc|experimental_config=|schedule=torch"


### 1.6 One step of each, same scale

Every bar is one program the TPU ran. Eager: a comb of small programs, one per op. DeferAndFuse: fewer, bigger blocks. Compiled: one. Same math in all three; the mode changes the shape, and the shape is the time.

In [ ]:
import glob, importlib.util, json, os

if importlib.util.find_spec("matplotlib") is None:
    %pip install -q matplotlib
import matplotlib.pyplot as plt


def tpu_programs(trace_dir, step_index=2):
    """The TPU programs of one step: [(start_ms, dur_ms, name)], and the step's length in ms."""
    path = max(glob.glob(f"{trace_dir}/*.pt.trace.json"), key=os.path.getmtime)
    with open(path) as f:
        events = json.load(f)["traceEvents"]
    threads = {(e["pid"], e["tid"]): e["args"]["name"]
               for e in events if e.get("ph") == "M" and e["name"] == "thread_name"}
    programs = [e for e in events if e.get("ph") == "X"
                and threads.get((e["pid"], e["tid"])) == "XLA Modules"]
    steps = sorted((e for e in events if e.get("ph") == "X"
                    and e["name"].startswith("ProfilerStep#")), key=lambda e: e["ts"])
    if steps:  # one steady step; the profiler marks each prof.step()
        s = steps[min(step_index, len(steps) - 1)]
        t0, t1 = s["ts"], s["ts"] + s["dur"]
    else:
        t0 = min(e["ts"] for e in programs)
        t1 = max(e["ts"] + e["dur"] for e in programs)
    inside = sorted((e for e in programs if t0 <= e["ts"] <= t1), key=lambda e: e["ts"])
    return [((e["ts"] - t0) / 1000, e["dur"] / 1000, e["name"]) for e in inside], (t1 - t0) / 1000


rows = [("eager", "#4285F4", *tpu_programs("traces/eager")),
        ("DeferAndFuse", "#F9AB00", *tpu_programs("traces/defer")),
        ("compiled", "#34A853", *tpu_programs("traces/compiled"))]
width = max(step_ms for _, _, _, step_ms in rows)

for label, _, bars, step_ms in rows:
    print(f"{label:13s} {len(bars)} program{'s' if len(bars) != 1 else ''} on the TPU in one step of {step_ms:.1f} ms")

fig, ax = plt.subplots(figsize=(14, 3.6))
for y, (label, color, bars, step_ms) in enumerate(reversed(rows)):
    ax.broken_barh([(s, max(d, width / 1400)) for s, d, _ in bars], (y + 0.15, 0.7),
                   facecolor=color, edgecolor="white", linewidth=0.8)
    n = f"{len(bars)} program{'s' if len(bars) != 1 else ''}"
    ax.text(width * 1.01, y + 0.5, f"{n}, {step_ms:.1f} ms", va="center")
ax.set_xlim(0, width)
ax.set_yticks([0.5, 1.5, 2.5], [r[0] for r in reversed(rows)])
ax.set_xlabel("ms from the start of the step (same scale for all three)")
ax.set_title("What the TPU ran in one step: one op at a time, fused blocks, or one program")
plt.tight_layout()
plt.show()

The session's version of the same picture, from the trace viewer (on a TPU VM: `pip install tensorboard xprof`, `tensorboard --logdir traces`, Profile tab, Trace Viewer):

![Profile results: one step of eager, DeferAndFuse and compiled, same scale](https://raw.githubusercontent.com/RobMulla/torchtpu-inference-workshop/main/notebooks/img/slide_05.jpg)

Debug in eager, speed up with one setting, ship compiled.

## Part 2 · SGLang on TorchTPU

![The KV cache: what makes serving an LLM different](https://raw.githubusercontent.com/RobMulla/torchtpu-inference-workshop/main/notebooks/img/slide_06.jpg)

### 2.1 Install

SGLang-TorchTPU gets a Python 3.12 environment of its own (its wheels are built for 3.12 and pin their own torch and TorchTPU); the notebook stays the client. The install lines are the project's README: two release wheels.

In [ ]:
import importlib.util, os, shutil, subprocess, sys

VENV = os.path.abspath("venv-sglang")  # the server's own Python 3.12 environment
PY = os.path.join(VENV, "bin", "python")
REL = "https://github.com/sgl-project/sglang-torchtpu/releases/download/tpu-v0.1.0"

if shutil.which("uv") is None:
    %pip install -q uv
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")
if not os.path.exists(PY):
    print("making", VENV, "and installing SGLang-TorchTPU into it (a few minutes the first time)")
    subprocess.run([UV, "venv", "-q", "--python", "3.12", VENV], check=True)
    subprocess.run([UV, "pip", "install", "-q", "--python", PY,
                    "--index-strategy", "unsafe-best-match",  # the best version across both indexes
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu",
                    f"{REL}/sglang-0.5.22-py3-none-any.whl",
                    f"{REL}/sglang_tpu-0.1.0-py3-none-any.whl", "tpu-info"], check=True)
%pip install -q openai requests

print(subprocess.run([PY, "-c", "import importlib.metadata as md, sys\n"
                      "print('python      ', sys.version.split()[0])\n"
                      "for p in ['sglang', 'sglang-tpu', 'torch', 'torch_tpu', 'libtpu']:\n"
                      "    print(f'{p:12s}', md.version(p))"],
                     capture_output=True, text=True).stdout)

assert subprocess.run([PY, "-c", "import sglang, sglang_tpu"], capture_output=True).returncode == 0, (
    "SGLang-TorchTPU is not in the environment (an empty placeholder install?): rm -rf " + VENV + " and run this cell again")

### 2.2 Start the server

`sglang.launch_server --device tpu`, the live demo's flags sized to one chip: `--tp-size 1`, an 8k context, `--enable-torch-compile` (the TPU opt-in), `--enable-cache-report` (so each response says how many prompt tokens came from the cache). It runs in the background; the log is `sglang_server.log`.

In [ ]:
import time

MODEL = "Qwen/Qwen3-0.6B"
PORT = 30000
LOG = os.path.abspath("sglang_server.log")
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

env = dict(os.environ,
           PYTHONUNBUFFERED="1",
           SGLANG_PLATFORM="tpu",
           SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN="1",
           TORCH_TPU_INTERNAL_MATERIALIZE_COLLECTIVE_TENSORS="false")

# Colab's sandbox hides /sys/bus/pci, so torch_tpu's device scan finds no TPU there; a
# sitecustomize on the server's PYTHONPATH registers the device anyway (no-op on a VM).
FIX = os.path.abspath("tpu_sitecustomize")
os.makedirs(FIX, exist_ok=True)
with open(os.path.join(FIX, "sitecustomize.py"), "w") as f:
    f.write('# The server\'s own Python 3.12 environment runs with this on PYTHONPATH. Colab\'s sandbox\n# hides /sys/bus/pci, so torch_tpu\'s device scan finds no TPU and never registers the\n# "tpu" device; this registers it anyway. On a TPU VM it does nothing.\ntry:\n    import torch, torch_tpu\n    from torch_tpu import _loader\n    if not _loader._LOADED:\n        _loader._init_device("tpu")\nexcept Exception:\n    pass\n')
env["PYTHONPATH"] = FIX + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

cmd = [PY, "-m", "sglang.launch_server",
       "--model-path", MODEL, "--device", "tpu",
       "--tp-size", "1",
       "--host", "127.0.0.1", "--port", str(PORT),
       "--context-length", "8192", "--max-running-requests", "8",
       "--page-size", "128", "--mem-fraction-static", "0.85",
       "--enable-torch-compile",
       "--enable-cache-report"]
print("python " + " ".join(cmd[1:]))
proc = subprocess.Popen(cmd, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
                        stdin=subprocess.DEVNULL, start_new_session=True)
print("pid", proc.pid, "| log", LOG)

### 2.3 Wait until it answers

`/health` returns 200 once the model is loaded and compiled. The first start also downloads the weights.

In [ ]:
import requests

t0 = time.time()
ready, last_print = False, -30
while time.time() - t0 < 1800:
    if proc.poll() is not None:
        break
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            ready = True
            break
    except requests.RequestException:  # refused while starting, or /health blocking on a compile
        pass
    if time.time() - t0 - last_print >= 30:
        last_print = time.time() - t0
        with open(LOG) as f:
            lines = f.read().splitlines()
        print(f"[{last_print:5.0f} s] {lines[-1][:110] if lines else ''}")
    time.sleep(1)

START_S = time.time() - t0
if ready:
    print(f"server ready after {START_S:.0f} s")
else:
    print("server not ready; last 40 log lines:")
    with open(LOG) as f:
        print("".join(f.readlines()[-40:]))

### 2.4 The chip

`tpu-info`: the server holds the chip, and most of its HBM is the model plus the KV cache the engine reserved.

In [ ]:
tpu_info = os.path.join(VENV, "bin", "tpu-info")


def show_chip():  # the chip table and the HBM table; the rest is multi-slice metrics
    out = subprocess.run([tpu_info], capture_output=True, text=True).stdout
    print(out.split("TensorCore Utilization")[0].rstrip())


show_chip()

### 2.5 One request

The standard `openai` client against the server's endpoint. It prints the time to the first token and the token counts.

In [ ]:
from openai import OpenAI

llm = OpenAI(base_url=BASE_URL, api_key="none")

t0 = time.monotonic()
first = None
stream = llm.chat.completions.create(
    model=MODEL, max_tokens=128,
    messages=[{"role": "user", "content": "In two sentences, what is a KV cache?"}],
    stream=True, stream_options={"include_usage": True},
    extra_body={"chat_template_kwargs": {"enable_thinking": False}})
for chunk in stream:
    for c in chunk.choices:
        if first is None and c.delta.content:
            first = (time.monotonic() - t0) * 1000
        print(c.delta.content or "", end="")
u = chunk.usage  # the last chunk carries the usage
print(f"\n\nfirst token: {first:.0f} ms | prompt {u.prompt_tokens} tokens, "
      f"completion {u.completion_tokens} tokens, {time.monotonic() - t0:.1f} s total")

### 2.6 A burst of sixteen

Sixteen different questions within one second, all streamed. The engine batches them on the one chip (continuous batching: requests join and leave the batch as they arrive and finish). The total is this chip's, this model's, with these settings.

In [ ]:
import asyncio
from openai import AsyncOpenAI

N = 16
allm = AsyncOpenAI(base_url=BASE_URL, api_key="none")

asks = ["Explain {} in detail.", "Define {} in one sentence.",
        "List five facts about {}.", "Explain {} in a short paragraph."]
topics = ["torch.compile", "autograd", "nn.Module", "DataLoader", "mixed precision",
          "LayerNorm", "the KV cache", "attention", "dropout", "Adam", "beam search",
          "top-p sampling", "RoPE", "quantization", "tokenizers", "batch norm"]


async def ask(i):
    await asyncio.sleep(i / N)  # arrivals spread over 1 s
    q = asks[i % 4].format(topics[i % 16])
    t0 = time.monotonic()
    stream = await allm.chat.completions.create(
        model=MODEL, max_tokens=256,
        messages=[{"role": "user", "content": q}],
        stream=True, stream_options={"include_usage": True},
        extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    usage = None
    async for chunk in stream:
        if chunk.usage:
            usage = chunk.usage
    dt = time.monotonic() - t0
    print(f"request {i:2d}: {usage.completion_tokens:4d} tokens in {dt:5.1f} s   ({q})")
    return usage.completion_tokens


t_burst = time.monotonic()
tokens = await asyncio.gather(*(ask(i) for i in range(N)))
wall = time.monotonic() - t_burst
BURST_TOK_S = sum(tokens) / wall
print(f"\n{N} requests, {sum(tokens)} completion tokens in {wall:.1f} s "
      f"= {BURST_TOK_S:.0f} tokens/s on this chip")

### 2.7 The prefix cache

One long document, two questions. The second time the document's pages are already in the cache: watch `cached` go from 0 to nearly the whole prompt and the first token come back sooner. (The cache is kept in 128-token pages, so the count is a little short of the prompt.)

![Prefix caching: ask about the same document twice, compute once](https://raw.githubusercontent.com/RobMulla/torchtpu-inference-workshop/main/notebooks/img/slide_13.jpg)

In [ ]:
import inspect
import json.decoder

doc = f"{time.time()}\n" + inspect.getsource(json.decoder)
print(f"prompt: {len(doc):,} characters\n")

for q in ["What does JSONDecodeError record?", "What does raw_decode return?"]:
    print(">", q)
    t0, first = time.monotonic(), None
    stream = llm.chat.completions.create(
        model=MODEL, max_tokens=64,
        messages=[{"role": "user", "content": f"{doc}\n\nIn one sentence: {q}"}],
        stream=True, stream_options={"include_usage": True},
        extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    for chunk in stream:
        for c in chunk.choices:
            if first is None and c.delta.content:
                first = (time.monotonic() - t0) * 1000
            print(c.delta.content or "", end="")
    u = chunk.usage
    d = u.prompt_tokens_details  # None when nothing was cached
    cached = d.cached_tokens if d else 0
    print(f"\nfirst token: {first:.0f} ms, cached {cached:,} of {u.prompt_tokens:,} prompt tokens\n")

### 2.8 Stop the server

Frees the chip for part 3.

In [ ]:
import signal


def stop_server(proc, timeout=60):
    if proc.poll() is not None:
        print("already stopped, exit code", proc.returncode)
        return
    os.killpg(proc.pid, signal.SIGINT)
    t0 = time.time()
    while proc.poll() is None and time.time() - t0 < timeout:
        time.sleep(1)
    if proc.poll() is None:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
    print(f"stopped after {time.time() - t0:.0f} s, exit code {proc.returncode}")


stop_server(proc)
time.sleep(3)
show_chip()

## Part 3 · vLLM on TorchTPU

### 3.1 Install

vLLM-TorchTPU in a Python 3.12 environment of its own: `pip install vllm-torchtpu`, one package that brings a matching vLLM.

In [ ]:
import importlib.util, os, shutil, subprocess, sys

VENV = os.path.abspath("venv-vllm")  # the server's own Python 3.12 environment
PY = os.path.join(VENV, "bin", "python")

if shutil.which("uv") is None:
    %pip install -q uv
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")
if not os.path.exists(PY):
    print("making", VENV, "and installing vLLM-TorchTPU into it (a few minutes the first time)")
    subprocess.run([UV, "venv", "-q", "--python", "3.12", VENV], check=True)
    subprocess.run([UV, "pip", "install", "-q", "--python", PY,
                    "--index-strategy", "unsafe-best-match",  # the best version across both indexes
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu",
                    "vllm-torchtpu", "tpu-info"], check=True)
%pip install -q openai requests

print(subprocess.run([PY, "-c", "import importlib.metadata as md, sys\n"
                      "print('python            ', sys.version.split()[0])\n"
                      "for p in ['vllm-torchtpu', 'vllm-torchtpu-core', 'torch', 'torch_tpu', 'libtpu']:\n"
                      "    print(f'{p:18s}', md.version(p))"],
                     capture_output=True, text=True).stdout)

assert os.path.exists(os.path.join(VENV, "bin", "vllm")), (
    "vLLM-TorchTPU is not in the environment (an empty placeholder install?): rm -rf " + VENV + " and run this cell again")

### 3.2 Start the server

`vllm serve` with no TPU flag: the plugin registers itself. `--tensor-parallel-size 1` on one chip, `--max-num-seqs 32`, and `--enable-prompt-tokens-details` so each response reports its cached prompt tokens. The log is `vllm_server.log`.

In [ ]:
import json
import time

MODEL = "Qwen/Qwen3-0.6B"
PORT = 8000
LOG = os.path.abspath("vllm_server.log")
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

env = dict(os.environ,
           PYTHONUNBUFFERED="1",
           VLLM_WORKER_MULTIPROC_METHOD="spawn",
           TORCH_TPU_INTERNAL_MATERIALIZE_COLLECTIVE_TENSORS="false",
           TORCHINDUCTOR_AUTOGRAD_CACHE="0")
if not env.get("TPU_ACCELERATOR_TYPE"):  # e.g. v6e-1, v5litepod-1: the chip tpu-info sees
    chip, n = json.loads(subprocess.run(
        [PY, "-c", "import json; from tpu_info import device; c, n = device.get_local_chips(); "
                   "print(json.dumps([c.value.name, n]))"], capture_output=True, text=True).stdout)
    env["TPU_ACCELERATOR_TYPE"] = {"v5e": "v5litepod", "7x": "tpu7x"}.get(chip, chip) + f"-{n}"
print("TPU_ACCELERATOR_TYPE", env["TPU_ACCELERATOR_TYPE"])

# Colab's sandbox hides /sys/bus/pci, so torch_tpu's device scan finds no TPU there; a
# sitecustomize on the server's PYTHONPATH registers the device anyway (no-op on a VM).
FIX = os.path.abspath("tpu_sitecustomize")
os.makedirs(FIX, exist_ok=True)
with open(os.path.join(FIX, "sitecustomize.py"), "w") as f:
    f.write('# The server\'s own Python 3.12 environment runs with this on PYTHONPATH. Colab\'s sandbox\n# hides /sys/bus/pci, so torch_tpu\'s device scan finds no TPU and never registers the\n# "tpu" device; this registers it anyway. On a TPU VM it does nothing.\ntry:\n    import torch, torch_tpu\n    from torch_tpu import _loader\n    if not _loader._LOADED:\n        _loader._init_device("tpu")\nexcept Exception:\n    pass\n')
env["PYTHONPATH"] = FIX + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

cmd = [os.path.join(VENV, "bin", "vllm"), "serve", MODEL,
       "--tensor-parallel-size", "1",
       "--host", "127.0.0.1", "--port", str(PORT),
       "--max-model-len", "8192",
       "--max-num-seqs", "32",
       "--max-num-batched-tokens", "2048",
       "--async-scheduling",
       "--enable-prompt-tokens-details"]
print("vllm " + " ".join(cmd[1:]))
proc = subprocess.Popen(cmd, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
                        stdin=subprocess.DEVNULL, start_new_session=True)
print("pid", proc.pid, "| log", LOG)

### 3.3 Wait until it answers

In [ ]:
import requests

t0 = time.time()
ready, last_print = False, -30
while time.time() - t0 < 1800:
    if proc.poll() is not None:
        break
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            ready = True
            break
    except requests.RequestException:  # refused while starting, or /health blocking on a compile
        pass
    if time.time() - t0 - last_print >= 30:
        last_print = time.time() - t0
        with open(LOG) as f:
            lines = f.read().splitlines()
        print(f"[{last_print:5.0f} s] {lines[-1][:110] if lines else ''}")
    time.sleep(1)

START_S = time.time() - t0
if ready:
    print(f"server ready after {START_S:.0f} s")
else:
    print("server not ready; last 40 log lines:")
    with open(LOG) as f:
        print("".join(f.readlines()[-40:]))

### 3.4 The chip

In [ ]:
tpu_info = os.path.join(VENV, "bin", "tpu-info")


def show_chip():  # the chip table and the HBM table; the rest is multi-slice metrics
    out = subprocess.run([tpu_info], capture_output=True, text=True).stdout
    print(out.split("TensorCore Utilization")[0].rstrip())


show_chip()

### 3.5 One request

The standard `openai` client against the server's endpoint; it prints the time to the first token and the token counts. The first request after a start can take a few seconds, and so can the first burst: the engine compiles once for each new shape it sees, then reuses the program.


In [ ]:
from openai import OpenAI

llm = OpenAI(base_url=BASE_URL, api_key="none")

t0 = time.monotonic()
first = None
stream = llm.chat.completions.create(
    model=MODEL, max_tokens=128,
    messages=[{"role": "user", "content": "In two sentences, what is a KV cache?"}],
    stream=True, stream_options={"include_usage": True},
    extra_body={"chat_template_kwargs": {"enable_thinking": False}})
for chunk in stream:
    for c in chunk.choices:
        if first is None and c.delta.content:
            first = (time.monotonic() - t0) * 1000
        print(c.delta.content or "", end="")
u = chunk.usage  # the last chunk carries the usage
print(f"\n\nfirst token: {first:.0f} ms | prompt {u.prompt_tokens} tokens, "
      f"completion {u.completion_tokens} tokens, {time.monotonic() - t0:.1f} s total")

### 3.6 A burst of sixteen

The engine re-forms the batch every step: a request joins as soon as it arrives and leaves as soon as it finishes, and one weight read serves everyone in the step.

![Continuous batching: requests join and leave every step](https://raw.githubusercontent.com/RobMulla/torchtpu-inference-workshop/main/notebooks/img/slide_15.jpg)

In [ ]:
import asyncio
from openai import AsyncOpenAI

N = 16
allm = AsyncOpenAI(base_url=BASE_URL, api_key="none")

asks = ["Explain {} in detail.", "Define {} in one sentence.",
        "List five facts about {}.", "Explain {} in a short paragraph."]
topics = ["torch.compile", "autograd", "nn.Module", "DataLoader", "mixed precision",
          "LayerNorm", "the KV cache", "attention", "dropout", "Adam", "beam search",
          "top-p sampling", "RoPE", "quantization", "tokenizers", "batch norm"]


async def ask(i):
    await asyncio.sleep(i / N)  # arrivals spread over 1 s
    q = asks[i % 4].format(topics[i % 16])
    t0 = time.monotonic()
    stream = await allm.chat.completions.create(
        model=MODEL, max_tokens=256,
        messages=[{"role": "user", "content": q}],
        stream=True, stream_options={"include_usage": True},
        extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    usage = None
    async for chunk in stream:
        if chunk.usage:
            usage = chunk.usage
    dt = time.monotonic() - t0
    print(f"request {i:2d}: {usage.completion_tokens:4d} tokens in {dt:5.1f} s   ({q})")
    return usage.completion_tokens


t_burst = time.monotonic()
tokens = await asyncio.gather(*(ask(i) for i in range(N)))
wall = time.monotonic() - t_burst
BURST_TOK_S = sum(tokens) / wall
print(f"\n{N} requests, {sum(tokens)} completion tokens in {wall:.1f} s "
      f"= {BURST_TOK_S:.0f} tokens/s on this chip")

### 3.7 The prefix cache

The same two questions about the same document; the second one finds it in the cache.

In [ ]:
import inspect
import json.decoder

doc = f"{time.time()}\n" + inspect.getsource(json.decoder)
print(f"prompt: {len(doc):,} characters\n")

for q in ["What does JSONDecodeError record?", "What does raw_decode return?"]:
    print(">", q)
    t0, first = time.monotonic(), None
    stream = llm.chat.completions.create(
        model=MODEL, max_tokens=64,
        messages=[{"role": "user", "content": f"{doc}\n\nIn one sentence: {q}"}],
        stream=True, stream_options={"include_usage": True},
        extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    for chunk in stream:
        for c in chunk.choices:
            if first is None and c.delta.content:
                first = (time.monotonic() - t0) * 1000
            print(c.delta.content or "", end="")
    u = chunk.usage
    d = u.prompt_tokens_details  # None when nothing was cached
    cached = d.cached_tokens if d else 0
    print(f"\nfirst token: {first:.0f} ms, cached {cached:,} of {u.prompt_tokens:,} prompt tokens\n")

### 3.8 Stop the server

In [ ]:
import signal


def stop_server(proc, timeout=60):
    if proc.poll() is not None:
        print("already stopped, exit code", proc.returncode)
        return
    os.killpg(proc.pid, signal.SIGINT)
    t0 = time.time()
    while proc.poll() is None and time.time() - t0 < timeout:
        time.sleep(1)
    if proc.poll() is None:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
    print(f"stopped after {time.time() - t0:.0f} s, exit code {proc.returncode}")


stop_server(proc)
time.sleep(3)
show_chip()

## Next

- TorchTPU: [github.com/google-pytorch/torch_tpu](https://github.com/google-pytorch/torch_tpu).
- vLLM on TPU: [github.com/vllm-project/vllm-torchtpu](https://github.com/vllm-project/vllm-torchtpu). SGLang on TPU: [github.com/sgl-project/sglang-torchtpu](https://github.com/sgl-project/sglang-torchtpu).
- The live demos' own files, with the exact commands for a four-chip VM: [`demos/`](https://github.com/RobMulla/torchtpu-inference-workshop/tree/main/demos).